# 03. Preprocesamiento de Interacciones

A partir del notebook de referencia, este notebook deja listas las interacciones para modelos de recomendación y evaluación temporal.


In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

from src.utils.recsys_utils import (
    add_indices,
    build_interactions,
    create_index_maps,
    ensure_dir,
    filter_min_interactions,
    load_clean_data,
    popularity_ranking,
    temporal_split,
)

_, movielens = load_clean_data()
processed_dir = ensure_dir(ROOT / 'data' / 'processed')
tables_dir = ensure_dir(ROOT / 'reports' / 'tables')


## Construir interacciones limpias

Partimos de `ratings.csv`, aplicamos filtros mínimos y preparamos un split temporal global.


In [ ]:
interactions = build_interactions(movielens)
raw_summary = pd.DataFrame([
    {'metric': 'raw_rows', 'value': len(interactions)},
    {'metric': 'raw_users', 'value': interactions['userId'].nunique()},
    {'metric': 'raw_items', 'value': interactions['movieId'].nunique()},
])
display(raw_summary)

filtered = filter_min_interactions(interactions, min_user_ratings=20, min_item_ratings=20, iterations=3)
filtered_summary = pd.DataFrame([
    {'metric': 'filtered_rows', 'value': len(filtered)},
    {'metric': 'filtered_users', 'value': filtered['userId'].nunique()},
    {'metric': 'filtered_items', 'value': filtered['movieId'].nunique()},
])
display(filtered_summary)


## Split temporal y mapeos de índices

El split respeta el orden temporal global para evitar leakage en la evaluación.


In [ ]:
train, valid, test = temporal_split(filtered, train_ratio=0.8, valid_ratio=0.1)
user_index, item_index = create_index_maps(train)

train_idx = add_indices(train, user_index, item_index)
valid_idx = add_indices(valid[valid['userId'].isin(user_index['userId']) & valid['movieId'].isin(item_index['movieId'])], user_index, item_index)
test_idx = add_indices(test[test['userId'].isin(user_index['userId']) & test['movieId'].isin(item_index['movieId'])], user_index, item_index)

split_summary = pd.DataFrame([
    {'split': 'train', 'rows': len(train_idx), 'users': train_idx['userId'].nunique(), 'items': train_idx['movieId'].nunique()},
    {'split': 'valid', 'rows': len(valid_idx), 'users': valid_idx['userId'].nunique(), 'items': valid_idx['movieId'].nunique()},
    {'split': 'test', 'rows': len(test_idx), 'users': test_idx['userId'].nunique(), 'items': test_idx['movieId'].nunique()},
])
display(split_summary)


## Baseline de popularidad y persistencia de artefactos

Además del split, guardamos un baseline simple para comparar luego contra CF e híbrido.


In [ ]:
popularity = popularity_ranking(train_idx)
display(popularity.head(10))

train_idx.to_parquet(processed_dir / 'interactions_train.parquet', index=False)
valid_idx.to_parquet(processed_dir / 'interactions_valid.parquet', index=False)
test_idx.to_parquet(processed_dir / 'interactions_test.parquet', index=False)
user_index.to_parquet(processed_dir / 'user_index.parquet', index=False)
item_index.to_parquet(processed_dir / 'item_index.parquet', index=False)
popularity.to_parquet(processed_dir / 'popularity_baseline.parquet', index=False)

preprocessing_summary = pd.concat([raw_summary, filtered_summary], ignore_index=True)
preprocessing_summary.to_csv(tables_dir / 'interaction_preprocessing_summary.csv', index=False)
split_summary.to_csv(tables_dir / 'interaction_split_summary.csv', index=False)
